# West Wing Power Dynamics: DeBERTa Multi-Task Training

Use this notebook on a Colab GPU runtime. It trains a shared DeBERTa encoder with three heads: power rating, power shift, and power strategies.

Minimal Google Drive upload:

- `WestWingPower/modeling/dataset.json`
- `WestWingPower/modeling/data.py`
- `WestWingPower/modeling/deberta_train.py`
- `WestWingPower/modeling/train.py`
- `WestWingPower/modeling/predict.py`
- `WestWingPower/modeling/__init__.py`
- `WestWingPower/modeling/requirements.txt`

Only upload `annotations/all_annotations.json` and the full `dialogues/` folder if you want to rebuild `dataset.json` inside Colab.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Change this if you upload the folder somewhere else in Drive.
PROJECT_DIR = '/content/drive/MyDrive/Annotation for NLP/WestWing'
%cd $PROJECT_DIR

/content/drive/MyDrive/Annotation for NLP/WestWing


In [4]:
!nvidia-smi
!python --version

Thu May  7 00:01:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!pip -q install -r modeling/requirements.txt

## Optional: Rebuild Dataset

Skip this cell if you uploaded `modeling/dataset.json`. Run it only if Drive also contains `annotations/all_annotations.json` and `dialogues/season_*/*.txt`.

In [ ]:
# !python modeling/data.py --annotations annotations/all_annotations.json --dialogues dialogues --output modeling/dataset.json

## Baseline Check

This re-runs the TF-IDF/logistic regression baseline and saves comparable prediction rows.

In [6]:
!python modeling/train.py \
  --dataset modeling/dataset.json \
  --model-name tfidf_logreg_baseline \
  --metrics-output modeling/artifacts/baseline_metrics.json \
  --predictions-output modeling/artifacts/baseline_predictions.csv


Power rating
              precision    recall  f1-score   support

          -2       0.30      0.27      0.29        11
          -1       0.24      0.18      0.21        22
           0       0.62      0.70      0.66        71
          +1       0.22      0.20      0.21        20
          +2       0.25      0.17      0.20         6

    accuracy                           0.48       130
   macro avg       0.32      0.31      0.31       130
weighted avg       0.45      0.48      0.46       130

accuracy=0.477 macro_f1=0.312

Power shift
              precision    recall  f1-score   support

           0       0.90      0.99      0.94       116
           1       0.50      0.07      0.12        14

    accuracy                           0.89       130
   macro avg       0.70      0.53      0.53       130
weighted avg       0.86      0.89      0.85       130

accuracy=0.892 macro_f1=0.534

Power strategies
                                 precision    recall  f1-score   support

  Dir

## DeBERTa Multi-Task Training

`microsoft/deberta-v3-base` is a good starting point. If Colab memory is tight, switch `--model-name` to `microsoft/deberta-v3-small` and keep the rest the same.

In [15]:
!python modeling/deberta_train.py \
  --dataset modeling/dataset.json \
  --model-name microsoft/deberta-v3-base \
  --run-name deberta_v3_base_multitask_continue \
  --output-dir modeling/artifacts/deberta_v3_base_multitask_continue \
  --max-length 384 \
  --batch-size 8 \
  --epochs 10 \
  --lr 5e-6 \
  --max-grad-norm 1.0 \
  --rating-loss-weight 1.0 \
  --shift-loss-weight 0.3 \
  --strategy-loss-weight 0.5 \
  --init-from-model modeling/artifacts/deberta_v3_base_multitask/best_model.pt

Using device: cuda
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 198/198 [00:00<00:00, 312.04it/s, Materializing param=encoder.rel_embeddings.weight]                     
DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight 

## Inspect Saved Outputs

In [13]:
import pandas as pd

baseline = pd.read_csv('modeling/artifacts/baseline_predictions.csv')
deberta = pd.read_csv('modeling/artifacts/deberta_v3_base_multitask/predictions.csv')
print(baseline.shape, deberta.shape)
deberta.head()

(648, 16) (648, 16)


,model_name,split,doc_id,episode,character_a,character_b,true_power_rating,pred_power_rating,power_rating_probs,true_power_shift,pred_power_shift,power_shift_probs,true_power_strategies,pred_power_strategies,power_strategy_probs,text
0,deberta_v3_base_multitask,test,S01E01_BILLY-SAM_pair1_exc1,S01E01,BILLY,SAM,0,0,"{""-2"": 0.19160030782222748, ""-1"": 0.2121903151...",0,0,"{""0"": 0.8265249133110046, ""1"": 0.1734751164913...",[],"[""Direct orders or instructions"", ""Interrogate...","{""Direct orders or instructions"": 0.5120740532...",SAM SEABORN\nI don't think we're going to run ...
1,deberta_v3_base_multitask,train,S01E01_CATHY-SAM_pair1_exc1,S01E01,CATHY,SAM,0,0,"{""-2"": 0.1867692619562149, ""-1"": 0.21586056053...",0,0,"{""0"": 0.8568677306175232, ""1"": 0.1431323289871...",[],"[""Interrogates or corners"", ""Appeals to author...","{""Direct orders or instructions"": 0.4986193776...","CATHY\nYeah, hi. [to Sam] I need you for just ..."
2,deberta_v3_base_multitask,test,S01E01_CATHY-SAM_pair2_exc1,S01E01,CATHY,SAM,0,-1,"{""-2"": 0.20414860546588898, ""-1"": 0.2374128252...",0,0,"{""0"": 0.8140771985054016, ""1"": 0.1859227567911...",[],"[""Interrogates or corners"", ""Emotional pressur...","{""Direct orders or instructions"": 0.4991669356...",CATHY\nYou're late.\n\nSAM\nI'm having kind of...
3,deberta_v3_base_multitask,test,S01E01_DONNA-JOSH_pair1_exc2,S01E01,DONNA,JOSH,1,0,"{""-2"": 0.19637885689735413, ""-1"": 0.2087651342...",0,0,"{""0"": 0.8343518972396851, ""1"": 0.1656481623649...",[],"[""Direct orders or instructions"", ""Interrogate...","{""Direct orders or instructions"": 0.5098413228...",JOSH\nClose the door. [Donna sets the coffee o...
4,deberta_v3_base_multitask,train,S01E01_DONNA-JOSH_pair2_exc1,S01E01,DONNA,JOSH,-1,0,"{""-2"": 0.17507782578468323, ""-1"": 0.1847258955...",0,0,"{""0"": 0.8621318936347961, ""1"": 0.1378681361675...","[""Manages or caretakes""]","[""Direct orders or instructions"", ""Interrogate...","{""Direct orders or instructions"": 0.5329766273...",JOSH\nNo.\n\nDONNA\nPut it on.\n\nJOSH\nNo.\n\...
